# Exercise 5 — Risk-Filtered Backtest Comparison

Apply `RiskManager` to an always-long strategy and compare it to the unfiltered version. The risk-filtered strategy should have a lower (less negative) max drawdown — that is the point of risk management. It may also have lower total return, but the Sharpe ratio might be higher.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def kelly_fraction(win_rate, avg_win, avg_loss):
    if avg_loss <= 0 or win_rate <= 0 or win_rate >= 1:
        return 0.0
    b = avg_win / avg_loss
    return max(0.0, min(1.0, win_rate - (1 - win_rate) / b))
def is_stopped_out(entry_price, current_price, stop_pct=0.05):
    if entry_price <= 0:
        return False
    return current_price <= entry_price * (1.0 - stop_pct)
def apply_stop_loss(signals, prices, stop_pct=0.05):
    result = signals.copy().astype(float)
    entry_price = None
    for i in range(len(result)):
        if result.iloc[i] == 1:
            if entry_price is None:
                entry_price = float(prices.iloc[i])
            elif is_stopped_out(entry_price, float(prices.iloc[i]), stop_pct):
                result.iloc[i] = 0
                entry_price = None
        else:
            entry_price = None
    return result.astype(int)
def market_drawdown(prices):
    peak = prices.cummax()
    return (prices - peak) / peak

def apply_drawdown_limit(signals, prices, limit=-0.20):
    dd = market_drawdown(prices)
    result = signals.copy().astype(int)
    result[dd < limit] = 0
    return result
class RiskManager:
    def __init__(self, stop_pct=0.05, drawdown_limit=-0.20):
        self.stop_pct = stop_pct
        self.drawdown_limit = drawdown_limit

    def filter(self, signals, prices):
        s = apply_stop_loss(signals, prices, self.stop_pct)
        return apply_drawdown_limit(s, prices, self.drawdown_limit)

    def summary(self, original, filtered):
        n_orig = int((original == 1).sum())
        n_kept = int((filtered == 1).sum())
        return {
            "total_long_bars": n_orig,
            "kept_long_bars":  n_kept,
            "filtered_bars":   n_orig - n_kept,
            "filter_rate":     float((n_orig - n_kept) / max(n_orig, 1)),
        }
def _compute_returns(df):  return df["Close"].pct_change()
def _compute_equity(r):    return (1 + r.fillna(0)).cumprod()
def _max_dd(eq):
    peak = eq.cummax(); return float(((eq - peak) / peak).min())
def _sharpe(r):
    c = r.dropna()
    if len(c) == 0 or c.std() == 0: return 0.0
    return float(c.mean() / c.std() * (252 ** 0.5))
def run_backtest(df, signals, label=""):
    mr  = _compute_returns(df)
    pos = signals.shift(1).fillna(0)
    sr  = pos * mr; eq = _compute_equity(sr)
    c   = sr.dropna(); n = len(c); tr = float(eq.iloc[-1] - 1.0)
    base = 1.0 + tr
    ar  = float(base ** (252.0 / max(n, 1)) - 1) if base > 0 else -1.0
    return {
        "label":             label,
        "total_return":      tr,
        "annualized_return": ar,
        "sharpe_ratio":      _sharpe(sr),
        "max_drawdown":      _max_dd(eq),
        "win_rate":          float((c > 0).sum() / max(n, 1)),
        "n_trades":          int((pos.diff().fillna(0) != 0).sum()),
        "equity":            eq,
    }


### Build strategies and run backtests

In [ ]:
df = _synthetic(n=252)

# Raw strategy: always long
sig_raw = pd.Series(1, index=df.index)

# Risk-filtered: apply stop-loss + drawdown limit
rm       = RiskManager(stop_pct=0.05, drawdown_limit=-0.20)
sig_risk = rm.filter(sig_raw, df["Close"])

# Summary of filtering
info = rm.summary(sig_raw, sig_risk)
print(f"Bars filtered: {info['filtered_bars']} / {info['total_long_bars']} "
      f"({info['filter_rate']:.2%} filter rate)")

r_raw  = run_backtest(df, sig_raw,  "Always-Long")
r_risk = run_backtest(df, sig_risk, "Risk-Filtered")


### Checks

In [ ]:
checks = 0

# 1 — risk-filtered has lower or equal max drawdown magnitude
try:
    # max_drawdown is negative; less negative = less severe
    assert r_risk["max_drawdown"] >= r_raw["max_drawdown"],         f"risk-filtered dd ({r_risk['max_drawdown']:.2%}) should be ≥ raw ({r_raw['max_drawdown']:.2%})"
    checks += 1; print("✅ 1 risk-filtered max drawdown is less severe than unfiltered")
except Exception as e:
    print("❌ 1:", e)

# 2 — risk-filtered has fewer or equal long bars (conservative)
try:
    assert sig_risk.sum() <= sig_raw.sum(),         f"filtered {sig_risk.sum()} 1s should be ≤ raw {sig_raw.sum()}"
    checks += 1; print("✅ 2 risk-filtered has fewer long bars than always-long")
except Exception as e:
    print("❌ 2:", e)

# 3 — equity consistency
try:
    for label, r in [("raw", r_raw), ("risk", r_risk)]:
        diff = abs(r["equity"].iloc[-1] - (1 + r["total_return"]))
        assert diff < 1e-9, f"{label}: equity[-1] != 1 + total_return"
    checks += 1; print("✅ 3 equity[-1] == 1 + total_return for both strategies")
except Exception as e:
    print("❌ 3:", e)

# 4 — max_drawdown is ≤ 0 for both
try:
    assert r_raw["max_drawdown"]  <= 1e-9
    assert r_risk["max_drawdown"] <= 1e-9
    checks += 1; print("✅ 4 max_drawdown ≤ 0 for both strategies")
except Exception as e:
    print("❌ 4:", e)

# 5 — print comparison table
try:
    print(f"\n{'Metric':<22} {'Always-Long':>12} {'Risk-Filtered':>14}")
    print("-" * 50)
    for key, fmt in [("total_return",".2%"),("sharpe_ratio",".3f"),
                     ("max_drawdown",".2%"),("n_trades","d")]:
        v1, v2 = r_raw[key], r_risk[key]
        if fmt == "d":
            print(f"{key:<22} {v1:>12d} {v2:>14d}")
        else:
            print(f"{key:<22} {v1:>{12}{fmt}} {v2:>{14}{fmt}}")
    checks += 1; print("\n✅ 5 comparison table printed")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
